<a href="https://colab.research.google.com/github/slavyolov/crew_ai_stock_recommendations/blob/master/notebooks/Stocks_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!python --version

Python 3.12.13


# Clone the repository

In [2]:
!git clone https://github.com/slavyolov/crew_ai_stock_recommendations

Cloning into 'crew_ai_stock_recommendations'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (42/42), done.
Receiving objects: 100% (65/65), 11.68 KiB | 11.68 MiB/s, done.
Resolving deltas: 100% (20/20), done.
remote: Total 65 (delta 20), reused 56 (delta 11), pack-reused 0 (from 0)


### Fixing the `git clone` error and updating the repository

The `git clone` command failed because the directory already exists. To get the latest changes, we should navigate into the existing repository and use `git pull`.

In [3]:
import os

# Navigate to the repository directory
%cd /content/crew_ai_stock_recommendations

# Perform a git pull to fetch and merge the latest changes from the remote
!git pull

# Display the current working directory
print(f"Current working directory: {os.getcwd()}")

/content/crew_ai_stock_recommendations
Already up to date.
Current working directory: /content/crew_ai_stock_recommendations


In [4]:
!pip install -r requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.8/120.8 kB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.9/642.9 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.6/15.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Setup Ollama and Download model

In [5]:
# Install zstd, then Ollama
!apt-get update -qq && apt-get install zstd -qq
!curl -fsSL https://ollama.ai/install.sh | sh

# Pull model (takes 5-10min first time)
!ollama pull openhermes

# Start server with proper background handling
import subprocess
import time
import threading

def start_ollama_server():
    process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    print("Ollama server starting...")
    return process

ollama_process = start_ollama_server()
time.sleep(15)  # Wait for startup

# Test connection
import requests
try:
    response = requests.get("http://localhost:11434/api/tags")
    print(f"✅ Ollama ready: {response.json()}")
except:
    print("❌ Ollama not ready - wait longer or restart")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Error: could not connect to ollama server, run 'ollama serve' to start it
Ollam

In [6]:
!ollama pull openhermes
!ollama list  # verify


NAME                 ID              SIZE      MODIFIED               
openhermes:latest    95477a2659b7    4.1 GB    Less than a second ago    


In [22]:
import requests
print(requests.get("http://localhost:11434/api/tags").json()["models"])

[{'name': 'openhermes:latest', 'model': 'openhermes:latest', 'modified_at': '2026-04-20T07:39:38.871011573Z', 'size': 4108928574, 'digest': '95477a2659b7539758230498d6ea9f6bfa5aa51ffb3dea9f37c91cacbac459c1', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '7B', 'quantization_level': 'Q4_0'}}]


# Setup API Keys

In [7]:
# Env vars for local Ollama
import os
os.environ['OPENAI_API_BASE'] = 'http://localhost:11434/v1'
os.environ['OPENAI_API_KEY'] = 'ollama'
os.environ['OPENAI_MODEL_NAME'] = 'openhermes'

In [8]:
import os
from google.colab import userdata

# Retrieve API keys from Colab Secrets
os.environ['SEC_API_KEY'] = userdata.get('SEC_API_KEY')
os.environ['SERPER_API_KEY'] = userdata.get('SERPER_API_KEY')
os.environ['SERPAPI_API_KEY'] = userdata.get('SERPER_API_KEY')

# Run the process

In [9]:
!pip install -q google-search-results

  Preparing metadata (setup.py) ... done


In [10]:
needed = ["OPENAI_API_KEY", "SERPER_API_KEY", "SEC_API_KEY"]
missing = [k for k in needed if not os.getenv(k)]
if missing:
    raise SystemExit(f"Missing environment variables: {', '.join(missing)}")

In [11]:
!python src/colab_stock_crew/main.py \
    --ticker NKE \
    --company "Nike"

╭─────────────────────────── Crew Execution Started ───────────────────────────╮
│                                                                              │
│  Crew Execution Started                                                      │
│  Name: crew                                                                  │
│  ID: ab98d821-c595-4091-b3ca-98bd11c2eb5d                                    │
│  Tool Args:                                                                  │
│                                                                              │
│                                                                              │
╰──────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
├── 📋 Task: market_research_task (ID: af15275c-76da-4768-bc01-83a7b6b644d5)
│   Status: Executing Task...
└── 🧠 Thinking...
╭────────────────────────────── 🤖 Agent Started ──────────────────────────────╮
│                                                    

# Monitor the Costs / API Keys
- https://serpapi.com/dashboard
- https://aistudio.google.com/
- https://sec-api.io/profile
- https://platform.openai.com/settings/organization/usage